# Geospatial Feature Engineering & Haversine Distance Calculation
Raw spatial coordinates (`latitude` and `longitude`) contain non-linear spatial signals that standard linear models struggle to interpret directly.

This notebook demonstrates computing spherical great-circle distances using the Haversine formula and deriving structural housing density ratios on the California Housing dataset.

In [1]:
import numpy as np
import pandas as pd

### Load Dataset from Direct Raw URL

In [2]:
url = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"
df = pd.read_csv(url)

In [3]:
df.head(2)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY


In [4]:
df.shape

(20640, 10)

In [5]:
df.info

<bound method DataFrame.info of        longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0        -122.23     37.88                41.0        880.0           129.0   
1        -122.22     37.86                21.0       7099.0          1106.0   
2        -122.24     37.85                52.0       1467.0           190.0   
3        -122.25     37.85                52.0       1274.0           235.0   
4        -122.25     37.85                52.0       1627.0           280.0   
...          ...       ...                 ...          ...             ...   
20635    -121.09     39.48                25.0       1665.0           374.0   
20636    -121.21     39.49                18.0        697.0           150.0   
20637    -121.22     39.43                17.0       2254.0           485.0   
20638    -121.32     39.43                18.0       1860.0           409.0   
20639    -121.24     39.37                16.0       2785.0           616.0   

       population  

In [6]:
df.describe()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
count,20640.000000,20640.000000,20640.000000,20640.000000,20433.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,-119.569704,35.631861,28.639486,2635.763081,537.870553,1425.476744,499.539680,3.870671,206855.816909
std,2.003532,2.135952,12.585558,2181.615252,421.385070,1132.462122,382.329753,1.899822,115395.615874
min,-124.350000,32.540000,1.000000,2.000000,1.000000,3.000000,1.000000,0.499900,14999.000000
25%,-121.800000,33.930000,18.000000,1447.750000,296.000000,787.000000,280.000000,2.563400,119600.000000
50%,-118.490000,34.260000,29.000000,2127.000000,435.000000,1166.000000,409.000000,3.534800,179700.000000
75%,-118.010000,37.710000,37.000000,3148.000000,647.000000,1725.000000,605.000000,4.743250,264725.000000
max,-114.310000,41.950000,52.000000,39320.000000,6445.000000,35682.000000,6082.000000,15.000100,500001.000000


In [7]:
df.isnull().sum()

longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64

In [8]:
df.duplicated().sum()

np.int64(0)

### Vectorized Haversine Distance Calculation Function (in Kilometers)

In [14]:
def haversine_np(lon1,lat1,lon2,lat2):
   lon1,lat1,lon2,lat2 = map(np.radians,[lon1,lat1,lon2,lat2])
   dlon = lon2-lon1
   dlat = lat2-lat1
   a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
   c = 2 * np.arcsin(np.sqrt(a))
   km = 6367 * c
   return km

### Define Reference Coordinates (San Francisco & Los Angeles)

In [15]:
sf_lat, sf_lon = 37.7749, -122.4194
la_lat, la_lon = 34.0522, -118.2437

### Compute Spatial Distance Features

In [16]:
df['dist_to_sf'] = haversine_np(df['longitude'],df['latitude'],sf_lon,sf_lat)
df['dist_to_la'] = haversine_np(df['longitude'],df['latitude'],la_lon,la_lat)

### Synthesize Household Structural Density Features

In [17]:
df['rooms_per_household'] = df['total_rooms'] / df['households']
df['bedrooms_per_room'] = df['total_bedrooms'] / df['total_rooms']

In [19]:
df[['latitude', 'longitude', 'dist_to_sf', 'dist_to_la', 'rooms_per_household', 'median_house_value']].head()

,latitude,longitude,dist_to_sf,dist_to_la,rooms_per_household,median_house_value
0,37.88,-122.23,20.316778,556.181784,6.984127,452600.0
1,37.86,-122.22,19.895566,553.933882,6.238137,358500.0
2,37.85,-122.24,17.824192,554.264528,8.288136,352100.0
3,37.85,-122.25,17.053410,554.847706,5.817352,341300.0
4,37.85,-122.25,17.053410,554.847706,6.281853,342200.0
